In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/enhancer-detection/weak_100.txt
/kaggle/input/enhancer-detection/strong_742.txt
/kaggle/input/enhancer-detection/strong_100.txt
/kaggle/input/enhancer-detection/enhancer.ind.txt
/kaggle/input/enhancer-detection/enhancer.cv.txt
/kaggle/input/enhancer-detection/non.cv.txt
/kaggle/input/enhancer-detection/non.ind.txt
/kaggle/input/enhancer-detection/weak_742.txt
/kaggle/input/enhancer-detection-2/weak_100.txt
/kaggle/input/enhancer-detection-2/strong_742.txt
/kaggle/input/enhancer-detection-2/strong_100.txt
/kaggle/input/enhancer-detection-2/enhancer.ind.txt
/kaggle/input/enhancer-detection-2/enhancer.cv.txt
/kaggle/input/enhancer-detection-2/non.cv.txt
/kaggle/input/enhancer-detection-2/non.ind.txt
/kaggle/input/enhancer-detection-2/weak_742.txt


LONGFORMER

In [3]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import LongformerTokenizer, LongformerModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef, recall_score, roc_curve, auc
from collections import Counter
import random, re, time
from torch.cuda.amp import autocast, GradScaler
import seaborn as sns

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Utilities ---
def clean_sequence(seq):
    return re.sub('[^ATCG]', '', seq.upper())

def load_fasta_clean(path):
    with open(path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    return seq.translate(str.maketrans("ATCG", "TAGC"))[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

def augment_sequence(seq):
    bases = ['A','T','C','G']
    seq_list = list(seq)
    for i in range(len(seq)):
        if random.random() < 0.05:
            seq_list[i] = random.choice([b for b in bases if b != seq_list[i]])
    kmers = ["".join(seq_list[i:i+6]) for i in range(len(seq_list)-5)]
    random.shuffle(kmers)
    return " ".join(kmers)

# --- Dataset ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512, augment=False):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        if self.augment:
            if random.random() < 0.5:
                seq = augment_sequence(seq)
            else:
                seq = kmer_tokenize(seq)
        else:
            seq = kmer_tokenize(seq)
        encoding = self.tokenizer(seq, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

# --- Model: Longformer Only ---
class LongformerOnly(nn.Module):
    def __init__(self, longformer_name="allenai/longformer-base-4096", dropout=0.3):
        super().__init__()
        self.longformer = LongformerModel.from_pretrained(longformer_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.longformer.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.longformer(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output  # [batch_size, hidden_dim]
        logits = self.classifier(self.dropout(pooled_output)).squeeze(-1)
        return logits

# --- Evaluation & Plotting ---
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'SN': recall_score(y_true, y_pred),
        'SP': tn / (tn + fp),
        'ACC': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_prob),
        'CM': cm
    }

def plot_auc_curve(y_true, y_prob, stage, fold):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc_score(y_true, y_prob):.4f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{stage} - Fold {fold+1} ROC Curve')
    plt.legend(loc='lower right')
    plt.savefig(f'{stage}_fold{fold+1}_roc.png')
    plt.close()

def plot_loss_curve(losses, stage, fold):
    plt.figure()
    plt.plot(losses, label='Training Loss')
    plt.title(f'{stage} - Fold {fold+1} Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(f'{stage}_fold{fold+1}_loss.png')
    plt.close()

def plot_confusion_matrix(cm, stage, fold):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    plt.title(f'{stage} - Fold {fold+1} Confusion Matrix')
    plt.savefig(f'{stage}_fold{fold+1}_cm.png')
    plt.close()

# --- Data Prep ---
def prepare_stage_data(stage):
    if stage == "Stage1":
        enhancers = load_fasta_clean("/kaggle/input/enhancer-detection/enhancer.cv.txt")
        non_enhancers = load_fasta_clean("/kaggle/input/enhancer-detection/non.cv.txt")
        enhancers = enhancers[:len(non_enhancers)]
        X = enhancers + [reverse_complement(seq) for seq in enhancers] + \
            non_enhancers + [reverse_complement(seq) for seq in non_enhancers]
        y = [1]*len(enhancers)*2 + [0]*len(non_enhancers)*2
    else:
        strong = load_fasta_clean("/kaggle/input/enhancer-detection/strong_742.txt")
        weak = load_fasta_clean("/kaggle/input/enhancer-detection/weak_742.txt")
        max_len = max(len(strong), len(weak))
        strong = (strong * ((max_len // len(strong)) + 1))[:max_len]
        weak = (weak * ((max_len // len(weak)) + 1))[:max_len]
        X = strong + [reverse_complement(seq) for seq in strong] + \
            weak + [reverse_complement(seq) for seq in weak]
        y = [1]*len(strong)*2 + [0]*len(weak)*2
    return X, y

# --- Training ---
def run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔁 {stage} - Fold {fold+1}/{k}")
        train_dataset = SimpleDataset([X[i] for i in train_idx], [y[i] for i in train_idx], tokenizer, augment=True)
        val_dataset = SimpleDataset([X[i] for i in val_idx], [y[i] for i in val_idx], tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=16 if stage=="Stage2" else 32, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=32)

        model = LongformerOnly()
        model = nn.DataParallel(model)
        model = model.to(device)

        optimizer = AdamW(model.parameters(), lr=1e-5 if stage == "Stage2" else 2e-5)
        loss_fn = nn.BCEWithLogitsLoss()
        scaler = GradScaler()

        losses = []
        start_time = time.time()

        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                with autocast():
                    logits = model(input_ids, attention_mask)
                    loss = loss_fn(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)
            losses.append(avg_loss)
            print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

        train_time = time.time() - start_time
        memory_used = torch.cuda.max_memory_allocated() / 1e6
        param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

        model.eval()
        y_true, y_pred, y_prob = [], [], []
        inference_start = time.time()
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                with autocast():
                    logits = model(input_ids, attention_mask)
                    probs = torch.sigmoid(logits)
                y_true.extend(labels.cpu().tolist())
                y_prob.extend(probs.cpu().tolist())
                y_pred.extend((probs > 0.5).int().cpu().tolist())
        inference_time = time.time() - inference_start

        metrics = compute_metrics(y_true, y_pred, y_prob)
        metrics.update({
            'TrainTimeSec': train_time,
            'InferTimeSec': inference_time,
            'Params': param_count,
            'MemMB': memory_used
        })
        print(metrics)

        plot_auc_curve(y_true, y_prob, stage, fold)
        plot_loss_curve(losses, stage, fold)
        plot_confusion_matrix(metrics['CM'], stage, fold)
        results.append(metrics)

    avg = {k: np.mean([fold[k] for fold in results if k in fold]) for k in results[0]}
    print(f"\n=== Final {stage} {k}-Fold Averages ===")
    for k, v in avg.items():
        print(f"{k}: {v:.4f}")

# --- Run ---
if __name__ == "__main__":
    tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
    for stage in ["Stage1", "Stage2"]:
        X, y = prepare_stage_data(stage)
        run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10)



🔁 Stage1 - Fold 1/10


pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

/tmp/ipykernel_36/425329467.py:176: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_36/425329467.py:190: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


model.safetensors:   0%|          | 0.00/597M [00:00<?, ?B/s]

Epoch 1 Loss: 0.5572
Epoch 2 Loss: 0.5284
Epoch 3 Loss: 0.5220
Epoch 4 Loss: 0.5339
Epoch 5 Loss: 0.5129


...

{'SN': 0.4555256064690027, 'SP': 0.633423180592992, 'ACC': 0.5444743935309974, 'F1': 0.5, 'MCC': 0.09039060677083512, 'AUC': 0.5868527546297979, 'CM': array([[235, 136],
       [202, 169]]), 'TrainTimeSec': 1970.9906203746796, 'InferTimeSec': 11.25967001914978, 'Params': 148660225, 'MemMB': 10774.418432}

🔁 Stage2 - Fold 10/10


/tmp/ipykernel_36/425329467.py:176: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_36/425329467.py:190: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 Loss: 0.6952
Epoch 2 Loss: 0.6941
Epoch 3 Loss: 0.6941
Epoch 4 Loss: 0.6950
Epoch 5 Loss: 0.6946


/tmp/ipykernel_36/425329467.py:214: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


{'SN': 0.9622641509433962, 'SP': 0.0673854447439353, 'ACC': 0.5148247978436657, 'F1': 0.664804469273743, 'MCC': 0.06643282473893374, 'AUC': 0.5325193801265611, 'CM': array([[ 25, 346],
       [ 14, 357]]), 'TrainTimeSec': 1968.8064863681793, 'InferTimeSec': 11.625539779663086, 'Params': 148660225, 'MemMB': 10774.418432}

=== Final Stage2 10-Fold Averages ===
SN: 0.5900
SP: 0.4647
ACC: 0.5274
F1: 0.5049
MCC: 0.0657
AUC: 0.5546
CM: 185.5000
TrainTimeSec: 1969.3977
InferTimeSec: 11.2981
Params: 148660225.0000
MemMB: 10774.4184


LONGFORMER + CNN:

In [2]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import LongformerTokenizer, LongformerModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef, recall_score, roc_curve
from collections import Counter
import random, re, time, os
import seaborn as sns
from torch.cuda.amp import autocast, GradScaler

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
set_seed()

# Utilities
def clean_sequence(seq):
    return re.sub('[^ATCG]', '', seq.upper())

def load_fasta_clean(path):
    with open(path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    return seq.translate(str.maketrans("ATCG", "TAGC"))[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

def random_mutation(seq, mutation_rate=0.05):
    bases = ['A','T','C','G']
    seq_list = list(seq)
    n_mut = max(1, int(len(seq)*mutation_rate))
    idxs = random.sample(range(len(seq)), n_mut)
    for i in idxs:
        seq_list[i] = random.choice([b for b in bases if b != seq_list[i]])
    return "".join(seq_list)

def kmer_shuffle(seq, k=6):
    kmers = [seq[i:i+k] for i in range(len(seq)-k+1)]
    random.shuffle(kmers)
    return " ".join(kmers)

def augment_sequence(seq):
    return kmer_shuffle(random_mutation(seq))

# Dataset
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512, augment=False):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        if self.augment and random.random() < 0.5:
            seq = augment_sequence(seq)
        else:
            seq = kmer_tokenize(seq)
        encoding = self.tokenizer(seq, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

# Model
class LongformerCNN(nn.Module):
    def __init__(self, model_name="allenai/longformer-base-4096", dropout=0.3):
        super().__init__()
        self.longformer = LongformerModel.from_pretrained(model_name)
        self.cnn = nn.Sequential(
            nn.Conv1d(self.longformer.config.hidden_size, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(64, 1)

    def forward(self, input_ids, attention_mask):
        output = self.longformer(input_ids=input_ids, attention_mask=attention_mask)
        hidden = output.last_hidden_state  # [B, T, H]
        cnn_input = hidden.permute(0, 2, 1)  # [B, H, T]
        cnn_out = self.cnn(cnn_input)       # [B, C, T']
        pooled = torch.max(cnn_out, dim=2)[0]  # [B, C]
        logits = self.classifier(self.dropout(pooled)).squeeze(-1)
        return logits

# Metrics & Plotting
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'SN': recall_score(y_true, y_pred),
        'SP': tn / (tn + fp),
        'ACC': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_prob),
        'CM': cm
    }

def plot_auc_curve(y_true, y_prob, stage, fold):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_true, y_prob):.4f}')
    plt.plot([0,1], [0,1], 'k--')
    plt.xlabel('FPR')
    plt.ylabel('TPR')
    plt.title(f'{stage} Fold {fold+1} ROC')
    plt.legend()
    plt.savefig(f"{stage}_fold{fold+1}_roc.png")
    plt.close()

def plot_loss_curve(losses, stage, fold):
    plt.figure()
    plt.plot(losses)
    plt.title(f'{stage} Fold {fold+1} Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.savefig(f"{stage}_fold{fold+1}_loss.png")
    plt.close()

def plot_confusion_matrix(cm, stage, fold):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{stage} Fold {fold+1} Confusion Matrix')
    plt.savefig(f"{stage}_fold{fold+1}_cm.png")
    plt.close()

# Data Prep
def prepare_stage_data(stage):
    if stage == "Stage1":
        enhancers = load_fasta_clean("/kaggle/input/enhancer-detection/enhancer.cv.txt")
        non_enhancers = load_fasta_clean("/kaggle/input/enhancer-detection/non.cv.txt")
        enhancers = enhancers[:len(non_enhancers)]
        X = enhancers + [reverse_complement(s) for s in enhancers] + \
            non_enhancers + [reverse_complement(s) for s in non_enhancers]
        y = [1]*len(enhancers)*2 + [0]*len(non_enhancers)*2
    else:
        strong = load_fasta_clean("/kaggle/input/enhancer-detection/strong_742.txt")
        weak = load_fasta_clean("/kaggle/input/enhancer-detection/weak_742.txt")
        max_len = max(len(strong), len(weak))
        strong = (strong * ((max_len // len(strong)) + 1))[:max_len]
        weak = (weak * ((max_len // len(weak)) + 1))[:max_len]
        X = strong + [reverse_complement(s) for s in strong] + \
            weak + [reverse_complement(s) for s in weak]
        y = [1]*len(strong)*2 + [0]*len(weak)*2
    return X, y

# Training Loop
def run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔁 {stage} - Fold {fold+1}/{k}")
        train_ds = SimpleDataset([X[i] for i in train_idx], [y[i] for i in train_idx], tokenizer, augment=True)
        val_ds = SimpleDataset([X[i] for i in val_idx], [y[i] for i in val_idx], tokenizer)
        train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=32)

        model = LongformerCNN()
        model = nn.DataParallel(model)
        model = model.to(device)

        optimizer = AdamW(model.parameters(), lr=1e-5)
        loss_fn = nn.BCEWithLogitsLoss()
        scaler = GradScaler()

        losses = []
        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                with autocast():
                    logits = model(input_ids, attention_mask)
                    loss = loss_fn(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item()
            avg_loss = total_loss / len(train_loader)
            losses.append(avg_loss)
            print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

        model.eval()
        y_true, y_pred, y_prob = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                with autocast():
                    probs = torch.sigmoid(model(input_ids, attention_mask))
                y_true.extend(labels.cpu().tolist())
                y_prob.extend(probs.cpu().tolist())
                y_pred.extend((probs > 0.5).int().cpu().tolist())

        y_prob = np.array(y_prob)
        if np.isnan(y_prob).any():
            print(f"🚨 NaNs in fold {fold+1}. Skipping.")
            continue

        metrics = compute_metrics(y_true, y_pred, y_prob)
        print(metrics)
        plot_auc_curve(y_true, y_prob, stage, fold)
        plot_loss_curve(losses, stage, fold)
        plot_confusion_matrix(metrics['CM'], stage, fold)
        results.append(metrics)

    avg = {k: np.mean([m[k] for m in results if k in m]) for k in results[0]}
    print(f"\n=== Final {stage} Results ===")
    for k, v in avg.items():
        print(f"{k}: {v:.4f}")

# Run
if __name__ == "__main__":
    tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
    for stage in ["Stage1", "Stage2"]:
        X, y = prepare_stage_data(stage)
        run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10)



🔁 Stage1 - Fold 1/10


/tmp/ipykernel_36/1716577230.py:185: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_36/1716577230.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 Loss: 0.5607
Epoch 2 Loss: 0.5358
Epoch 3 Loss: 0.5302
Epoch 4 Loss: 0.5245
Epoch 5 Loss: 0.5181


/tmp/ipykernel_36/1716577230.py:214: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


{'SN': 0.7306397306397306, 'SP': 0.7609427609427609, 'ACC': 0.7457912457912458, 'F1': 0.7418803418803418, 'MCC': 0.49180835076811963, 'AUC': 0.8268714076794885, 'CM': array([[226,  71],
       [ 80, 217]])}

🔁 Stage1 - Fold 2/10


...

{'SN': 0.9083557951482479, 'SP': 0.21293800539083557, 'ACC': 0.5606469002695418, 'F1': 0.674, 'MCC': 0.16879048819965736, 'AUC': 0.5848075791370304, 'CM': array([[ 79, 292],
       [ 34, 337]])}

🔁 Stage2 - Fold 10/10


/tmp/ipykernel_36/1716577230.py:185: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_36/1716577230.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 Loss: 0.6953
Epoch 2 Loss: 0.6859
Epoch 3 Loss: 0.6831
Epoch 4 Loss: 0.6817
Epoch 5 Loss: 0.6820


/tmp/ipykernel_36/1716577230.py:214: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


{'SN': 0.7628032345013477, 'SP': 0.33153638814016173, 'ACC': 0.5471698113207547, 'F1': 0.6274944567627495, 'MCC': 0.10456336623755626, 'AUC': 0.588360299620026, 'CM': array([[123, 248],
       [ 88, 283]])}

=== Final Stage2 Results ===
SN: 0.5886
SP: 0.4805
ACC: 0.5345
F1: 0.4701
MCC: 0.0798
AUC: 0.5518
CM: 185.5000


LONGFORMER + BiLSTM

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import LongformerTokenizer, LongformerModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef, recall_score, roc_curve
from collections import Counter
from tqdm import tqdm
import random, re, time
from torch.cuda.amp import autocast, GradScaler
import seaborn as sns

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# Utilities
def clean_sequence(seq):
    return re.sub('[^ATCG]', '', seq.upper())

def load_fasta_clean(path):
    with open(path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    return seq.translate(str.maketrans("ATCG", "TAGC"))[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

def augment_sequence(seq):
    bases = ['A','T','C','G']
    seq_list = list(seq)
    for i in range(len(seq)):
        if random.random() < 0.05:
            seq_list[i] = random.choice([b for b in bases if b != seq_list[i]])
    kmers = ["".join(seq_list[i:i+6]) for i in range(len(seq_list)-5)]
    random.shuffle(kmers)
    return " ".join(kmers)

# Dataset
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512, augment=False):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        if self.augment and random.random() < 0.5:
            seq = augment_sequence(seq)
        else:
            seq = kmer_tokenize(seq)
        encoding = self.tokenizer(seq, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

# Model: Longformer + BiLSTM
class Stage1Model(nn.Module):
    def __init__(self, longformer_model_name="allenai/longformer-base-4096", hidden_dim=128, dropout=0.3):
        super().__init__()
        self.longformer = LongformerModel.from_pretrained(longformer_model_name)
        self.lstm = nn.LSTM(self.longformer.config.hidden_size, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.longformer(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        lstm_out, _ = self.lstm(last_hidden)
        pooled = torch.mean(lstm_out, dim=1)
        logits = self.classifier(self.dropout(pooled)).squeeze(-1)
        return logits

# Metrics
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'SN': recall_score(y_true, y_pred),
        'SP': tn / (tn + fp),
        'ACC': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_prob),
        'CM': cm
    }

def plot_auc_curve(y_true, y_prob, stage, fold):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc_score(y_true, y_prob):.4f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{stage} - Fold {fold+1} ROC Curve')
    plt.legend(loc='lower right')
    plt.savefig(f'{stage}_fold{fold+1}_roc.png')
    plt.close()

def plot_loss_curve(losses, stage, fold):
    plt.figure()
    plt.plot(losses, label='Training Loss')
    plt.title(f'{stage} - Fold {fold+1} Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(f'{stage}_fold{fold+1}_loss.png')
    plt.close()

def plot_confusion_matrix(cm, stage, fold):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    plt.title(f'{stage} - Fold {fold+1} Confusion Matrix')
    plt.savefig(f'{stage}_fold{fold+1}_cm.png')
    plt.close()

# Data prep for Stage 1
def prepare_stage1_data():
    enhancers = load_fasta_clean("/kaggle/input/enhancer-detection/enhancer.cv.txt")
    non_enhancers = load_fasta_clean("/kaggle/input/enhancer-detection/non.cv.txt")
    enhancers = enhancers[:len(non_enhancers)]
    X = enhancers + [reverse_complement(seq) for seq in enhancers] + \
        non_enhancers + [reverse_complement(seq) for seq in non_enhancers]
    y = [1]*len(enhancers)*2 + [0]*len(non_enhancers)*2
    return X, y

# Training
def run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔁 {stage} - Fold {fold+1}/{k}")
        train_dataset = SimpleDataset([X[i] for i in train_idx], [y[i] for i in train_idx], tokenizer, augment=True)
        val_dataset = SimpleDataset([X[i] for i in val_idx], [y[i] for i in val_idx], tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=32)

        model = Stage1Model()
        model = nn.DataParallel(model)
        model = model.to(device)

        optimizer = AdamW(model.parameters(), lr=2e-5)
        loss_fn = nn.BCEWithLogitsLoss()
        scaler = GradScaler()

        losses = []
        start_time = time.time()

        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                with autocast():
                    logits = model(input_ids, attention_mask)
                    loss = loss_fn(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)
            losses.append(avg_loss)
            print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

        train_time = time.time() - start_time
        memory_used = torch.cuda.max_memory_allocated() / 1e6
        param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

        model.eval()
        y_true, y_pred, y_prob = [], [], []
        inference_start = time.time()
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                with autocast():
                    logits = model(input_ids, attention_mask)
                    probs = torch.sigmoid(logits)
                y_true.extend(labels.cpu().tolist())
                y_prob.extend(probs.cpu().tolist())
                y_pred.extend((probs > 0.5).int().cpu().tolist())
        inference_time = time.time() - inference_start

        metrics = compute_metrics(y_true, y_pred, y_prob)
        metrics.update({
            'TrainTimeSec': train_time,
            'InferTimeSec': inference_time,
            'Params': param_count,
            'MemMB': memory_used
        })
        print(metrics)

        plot_auc_curve(y_true, y_prob, stage, fold)
        plot_loss_curve(losses, stage, fold)
        plot_confusion_matrix(metrics['CM'], stage, fold)
        results.append(metrics)

    avg = {k: np.mean([fold[k] for fold in results if k in fold]) for k in results[0]}
    print(f"\n=== Final {stage} {k}-Fold Averages ===")
    for k, v in avg.items():
        print(f"{k}: {v:.4f}")

# Run
if __name__ == "__main__":
    tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
    X, y = prepare_stage1_data()
    run_cross_validation(X, y, "Stage1", tokenizer, epochs=5, k=10)


2025-07-15 14:27:42.171045: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752589662.368007      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752589662.425749      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

{'SN': 0.777027027027027, 'SP': 0.7373737373737373, 'ACC': 0.7571669477234402, 'F1': 0.7615894039735098, 'MCC': 0.5147873866007039, 'AUC': 0.8301142051142051, 'CM': array([[219,  78],
       [ 66, 230]]), 'TrainTimeSec': 1472.88547372818, 'InferTimeSec': 10.14783763885498, 'Params': 149579265, 'MemMB': 10881.588736}

🔁 Stage1 - Fold 10/10


/tmp/ipykernel_36/2443649642.py:167: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_36/2443649642.py:181: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 Loss: 0.5577
Epoch 2 Loss: 0.5284
Epoch 3 Loss: 0.5206
Epoch 4 Loss: 0.5134
Epoch 5 Loss: 0.5114


/tmp/ipykernel_36/2443649642.py:205: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


{'SN': 0.6722972972972973, 'SP': 0.8518518518518519, 'ACC': 0.7622259696458684, 'F1': 0.738404452690167, 'MCC': 0.5328951974998871, 'AUC': 0.8534329784329784, 'CM': array([[253,  44],
       [ 97, 199]]), 'TrainTimeSec': 1472.6582844257355, 'InferTimeSec': 10.148679733276367, 'Params': 149579265, 'MemMB': 10882.166784}

=== Final Stage1 10-Fold Averages ===
SN: 0.7237
SP: 0.7675
ACC: 0.7456
F1: 0.7390
MCC: 0.4952
AUC: 0.8296
CM: 148.4000
TrainTimeSec: 1473.9435
InferTimeSec: 10.2613
Params: 149579265.0000
MemMB: 10879.8632


In [2]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import LongformerTokenizer, LongformerModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef, recall_score
from collections import Counter
from tqdm import tqdm
import random, re, time
from torch.cuda.amp import autocast, GradScaler
import seaborn as sns

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Utilities ---
def clean_sequence(seq):
    return re.sub('[^ATCG]', '', seq.upper())

def load_fasta_clean(path):
    with open(path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    return seq.translate(str.maketrans("ATCG", "TAGC"))[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

def augment_sequence(seq):
    bases = ['A', 'T', 'C', 'G']
    seq_list = list(seq)
    for i in range(len(seq)):
        if random.random() < 0.05:
            seq_list[i] = random.choice([b for b in bases if b != seq_list[i]])
    kmers = ["".join(seq_list[i:i+6]) for i in range(len(seq_list)-5)]
    random.shuffle(kmers)
    return " ".join(kmers)

# --- Dataset ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512, augment=False):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        if self.augment:
            if random.random() < 0.5:
                seq = augment_sequence(seq)
            else:
                seq = kmer_tokenize(seq)
        else:
            seq = kmer_tokenize(seq)
        encoding = self.tokenizer(seq, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

# --- Stage 2 Variant 3: LONGFORMER + BiLSTM ---
class Stage2Model(nn.Module):
    def __init__(self, longformer_model_name="allenai/longformer-base-4096", hidden_dim=128, dropout=0.3):
        super().__init__()
        self.longformer = LongformerModel.from_pretrained(longformer_model_name)
        self.lstm = nn.LSTM(self.longformer.config.hidden_size, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.longformer(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        lstm_out, _ = self.lstm(last_hidden)
        pooled = torch.mean(lstm_out, dim=1)
        logits = self.classifier(self.dropout(pooled)).squeeze(-1)
        return logits

# --- Evaluation & Plotting ---
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'SN': recall_score(y_true, y_pred),
        'SP': tn / (tn + fp),
        'ACC': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_prob),
        'CM': cm
    }

def plot_auc_curve(y_true, y_prob, stage, fold):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc_score(y_true, y_prob):.4f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{stage} - Fold {fold+1} ROC Curve')
    plt.legend(loc='lower right')
    plt.savefig(f'{stage}_fold{fold+1}_roc.png')
    plt.close()

def plot_loss_curve(losses, stage, fold):
    plt.figure()
    plt.plot(losses, label='Training Loss')
    plt.title(f'{stage} - Fold {fold+1} Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(f'{stage}_fold{fold+1}_loss.png')
    plt.close()

def plot_confusion_matrix(cm, stage, fold):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    plt.title(f'{stage} - Fold {fold+1} Confusion Matrix')
    plt.savefig(f'{stage}_fold{fold+1}_cm.png')
    plt.close()

# --- Data Prep ---
def prepare_stage_data(stage):
    strong = load_fasta_clean("/kaggle/input/enhancer-detection/strong_742.txt")
    weak = load_fasta_clean("/kaggle/input/enhancer-detection/weak_742.txt")
    max_len = max(len(strong), len(weak))
    strong = (strong * ((max_len // len(strong)) + 1))[:max_len]
    weak = (weak * ((max_len // len(weak)) + 1))[:max_len]
    X = strong + [reverse_complement(seq) for seq in strong] + \
        weak + [reverse_complement(seq) for seq in weak]
    y = [1]*len(strong)*2 + [0]*len(weak)*2
    return X, y

# --- Training ---
def run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔁 {stage} - Fold {fold+1}/{k}")
        train_dataset = SimpleDataset([X[i] for i in train_idx], [y[i] for i in train_idx], tokenizer, augment=True)
        val_dataset = SimpleDataset([X[i] for i in val_idx], [y[i] for i in val_idx], tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=32)

        model = Stage2Model()
        model = nn.DataParallel(model)
        model = model.to(device)

        optimizer = AdamW(model.parameters(), lr=1e-5)
        loss_fn = nn.BCEWithLogitsLoss()
        scaler = GradScaler()

        losses = []
        start_time = time.time()
        torch.cuda.reset_peak_memory_stats()

        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                with autocast():
                    logits = model(input_ids, attention_mask)
                    loss = loss_fn(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)
            losses.append(avg_loss)
            print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

        train_time = time.time() - start_time
        memory_used = torch.cuda.max_memory_allocated() / 1e6
        param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

        model.eval()
        y_true, y_pred, y_prob = [], [], []
        inference_start = time.time()
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                with autocast():
                    logits = model(input_ids, attention_mask)
                    probs = torch.sigmoid(logits)
                y_true.extend(labels.cpu().tolist())
                y_prob.extend(probs.cpu().tolist())
                y_pred.extend((probs > 0.5).int().cpu().tolist())
        inference_time = time.time() - inference_start

        metrics = compute_metrics(y_true, y_pred, y_prob)
        metrics.update({
            'TrainTimeSec': train_time,
            'InferTimeSec': inference_time,
            'Params': param_count,
            'MemMB': memory_used
        })
        print(metrics)

        plot_auc_curve(y_true, y_prob, stage, fold)
        plot_loss_curve(losses, stage, fold)
        plot_confusion_matrix(metrics['CM'], stage, fold)
        results.append(metrics)

    avg = {k: np.mean([fold[k] for fold in results if k in fold]) for k in results[0]}
    print(f"\n=== Final {stage} {k}-Fold Averages ===")
    for k, v in avg.items():
        print(f"{k}: {v:.4f}")

# --- Run ---
if __name__ == "__main__":
    tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096", trust_remote_code=True)
    X, y = prepare_stage_data("Stage2")
    run_cross_validation(X, y, "Stage2", tokenizer, epochs=5, k=10)



🔁 Stage2 - Fold 1/10


pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/597M [00:00<?, ?B/s]

/tmp/ipykernel_36/354794587.py:173: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_36/354794587.py:188: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 Loss: 0.6899
Epoch 2 Loss: 0.6839
Epoch 3 Loss: 0.6820
Epoch 4 Loss: 0.6809
Epoch 5 Loss: 0.6812


...

{'SN': 0.8733153638814016, 'SP': 0.2749326145552561, 'ACC': 0.5741239892183289, 'F1': 0.6721991701244814, 'MCC': 0.18503002546926856, 'AUC': 0.6030652203921797, 'CM': array([[102, 269],
       [ 47, 324]]), 'TrainTimeSec': 1968.3011500835419, 'InferTimeSec': 11.321359395980835, 'Params': 149579265, 'MemMB': 6497.024512}

🔁 Stage2 - Fold 10/10


/tmp/ipykernel_36/354794587.py:173: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_36/354794587.py:188: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 Loss: 0.6916
Epoch 2 Loss: 0.6839
Epoch 3 Loss: 0.6828
Epoch 4 Loss: 0.6806
Epoch 5 Loss: 0.6803


/tmp/ipykernel_36/354794587.py:212: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


{'SN': 0.8274932614555256, 'SP': 0.29110512129380056, 'ACC': 0.5592991913746631, 'F1': 0.6524973432518597, 'MCC': 0.14052416882395644, 'AUC': 0.5972566313816378, 'CM': array([[108, 263],
       [ 64, 307]]), 'TrainTimeSec': 1969.2440357208252, 'InferTimeSec': 10.977473735809326, 'Params': 149579265, 'MemMB': 6495.840768}

=== Final Stage2 10-Fold Averages ===
SN: 0.6655
SP: 0.4170
ACC: 0.5412
F1: 0.5772
MCC: 0.0928
AUC: 0.5725
CM: 185.5000
TrainTimeSec: 1969.7518
InferTimeSec: 11.0644
Params: 149579265.0000
MemMB: 6496.6832


LONGFORMER + Transformer Encoder

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import LongformerTokenizer, LongformerModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef, recall_score, roc_curve, auc
from collections import Counter
import random, re, time
from torch.cuda.amp import autocast, GradScaler
import seaborn as sns
import os

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Utilities ---
def clean_sequence(seq):
    return re.sub('[^ATCG]', '', seq.upper())

def load_fasta_clean(path):
    with open(path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    return seq.translate(str.maketrans("ATCG", "TAGC"))[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

def augment_sequence(seq):
    bases = ['A','T','C','G']
    seq_list = list(seq)
    for i in range(len(seq)):
        if random.random() < 0.05:
            seq_list[i] = random.choice([b for b in bases if b != seq_list[i]])
    kmers = ["".join(seq_list[i:i+6]) for i in range(len(seq_list)-5)]
    random.shuffle(kmers)
    return " ".join(kmers)

# --- Dataset ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512, augment=False):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        if self.augment and random.random() < 0.5:
            seq = augment_sequence(seq)
        else:
            seq = kmer_tokenize(seq)
        encoding = self.tokenizer(seq, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

# --- Model: Longformer + Transformer Encoder ---
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_output, _ = self.attn(x, x, x)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.ff(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

class LongformerTransformerEncoder(nn.Module):
    def __init__(self, model_name="allenai/longformer-base-4096", dropout=0.3):
        super().__init__()
        self.longformer = LongformerModel.from_pretrained(model_name)
        self.transformer = TransformerEncoderBlock(self.longformer.config.hidden_size, num_heads=4, dropout=dropout)
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.longformer.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.longformer(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state  # [batch_size, seq_len, hidden_size]
        encoded = self.transformer(hidden_states)
        pooled = self.pooling(encoded.transpose(1, 2)).squeeze(-1)
        logits = self.classifier(self.dropout(pooled)).squeeze(-1)
        return logits

# --- Evaluation & Plotting ---
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'SN': recall_score(y_true, y_pred),
        'SP': tn / (tn + fp + 1e-8),
        'ACC': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_prob),
        'CM': cm
    }

def plot_auc_curve(y_true, y_prob, stage, fold):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc_score(y_true, y_prob):.4f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{stage} - Fold {fold+1} ROC Curve')
    plt.legend(loc='lower right')
    plt.savefig(f'{stage}_fold{fold+1}_roc.png')
    plt.close()

def plot_loss_curve(losses, stage, fold):
    plt.figure()
    plt.plot(losses, label='Training Loss')
    plt.title(f'{stage} - Fold {fold+1} Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(f'{stage}_fold{fold+1}_loss.png')
    plt.close()

def plot_confusion_matrix(cm, stage, fold):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    plt.title(f'{stage} - Fold {fold+1} Confusion Matrix')
    plt.savefig(f'{stage}_fold{fold+1}_cm.png')
    plt.close()

# --- Data Prep ---
def prepare_stage_data(stage):
    if stage == "Stage1":
        enhancers = load_fasta_clean("/kaggle/input/enhancer-detection/enhancer.cv.txt")
        non_enhancers = load_fasta_clean("/kaggle/input/enhancer-detection/non.cv.txt")
        enhancers = enhancers[:len(non_enhancers)]
        X = enhancers + [reverse_complement(seq) for seq in enhancers] + \
            non_enhancers + [reverse_complement(seq) for seq in non_enhancers]
        y = [1]*len(enhancers)*2 + [0]*len(non_enhancers)*2
    else:
        strong = load_fasta_clean("/kaggle/input/enhancer-detection/strong_742.txt")
        weak = load_fasta_clean("/kaggle/input/enhancer-detection/weak_742.txt")
        max_len = max(len(strong), len(weak))
        strong = (strong * ((max_len // len(strong)) + 1))[:max_len]
        weak = (weak * ((max_len // len(weak)) + 1))[:max_len]
        X = strong + [reverse_complement(seq) for seq in strong] + \
            weak + [reverse_complement(seq) for seq in weak]
        y = [1]*len(strong)*2 + [0]*len(weak)*2
    return X, y

# --- Training ---
def run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔁 {stage} - Fold {fold+1}/{k}")
        train_dataset = SimpleDataset([X[i] for i in train_idx], [y[i] for i in train_idx], tokenizer, augment=True)
        val_dataset = SimpleDataset([X[i] for i in val_idx], [y[i] for i in val_idx], tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=32)

        model = LongformerTransformerEncoder()
        model = nn.DataParallel(model)
        model = model.to(device)

        optimizer = AdamW(model.parameters(), lr=1e-5)
        loss_fn = nn.BCEWithLogitsLoss()
        scaler = GradScaler()

        losses = []
        start_time = time.time()

        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                with autocast():
                    logits = model(input_ids, attention_mask)
                    loss = loss_fn(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)
            losses.append(avg_loss)
            print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

        train_time = time.time() - start_time
        memory_used = torch.cuda.max_memory_allocated() / 1e6
        param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

        model.eval()
        y_true, y_pred, y_prob = [], [], []
        inference_start = time.time()
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                with autocast():
                    logits = model(input_ids, attention_mask)
                    probs = torch.sigmoid(logits)
                y_true.extend(labels.cpu().tolist())
                y_prob.extend(probs.cpu().tolist())
                y_pred.extend((probs > 0.5).int().cpu().tolist())
        inference_time = time.time() - inference_start

        metrics = compute_metrics(y_true, y_pred, y_prob)
        metrics.update({
            'TrainTimeSec': train_time,
            'InferTimeSec': inference_time,
            'Params': param_count,
            'MemMB': memory_used
        })
        print(metrics)

        plot_auc_curve(y_true, y_prob, stage, fold)
        plot_loss_curve(losses, stage, fold)
        plot_confusion_matrix(metrics['CM'], stage, fold)
        results.append(metrics)

    avg = {k: np.mean([fold[k] for fold in results if k in fold]) for k in results[0]}
    print(f"\n=== Final {stage} {k}-Fold Averages ===")
    for k, v in avg.items():
        print(f"{k}: {v:.4f}")

# --- Run ---
if __name__ == "__main__":
    tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
    for stage in ["Stage1", "Stage2"]:
        X, y = prepare_stage_data(stage)
        run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10)


2025-07-16 04:39:49.951920: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752640790.154002      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752640790.210290      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

/tmp/ipykernel_36/1684801877.py:198: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_36/1684801877.py:212: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 Loss: 0.5719
Epoch 2 Loss: 0.5344
Epoch 3 Loss: 0.5243
Epoch 4 Loss: 0.5324
Epoch 5 Loss: 0.5220


/tmp/ipykernel_36/1684801877.py:236: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


{'SN': 0.7702702702702703, 'SP': 0.72390572388135, 'ACC': 0.7470489038785835, 'F1': 0.7524752475247525, 'MCC': 0.4946883242401629, 'AUC': 0.8492924742924742, 'CM': array([[215,  82],
       [ 68, 228]]), 'TrainTimeSec': 1658.3459405899048, 'InferTimeSec': 10.3032808303833, 'Params': 155748097, 'MemMB': 6681.776128}

=== Final Stage1 10-Fold Averages ===
SN: 0.7311
SP: 0.7621
ACC: 0.7466
F1: 0.7411
MCC: 0.4987
AUC: 0.8302
CM: 148.4000
TrainTimeSec: 1658.7758
InferTimeSec: 10.2925
Params: 155748097.0000
MemMB: 6681.7761

🔁 Stage2 - Fold 1/10


/tmp/ipykernel_36/1684801877.py:198: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_36/1684801877.py:212: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef, recall_score
from collections import Counter
import random, re, os, gc, time
from torch.amp import autocast, GradScaler
import seaborn as sns

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Utilities ---
def clean_sequence(seq):
    return re.sub('[^ATCG]', '', seq.upper())

def load_fasta_clean(path):
    with open(path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    return seq.translate(str.maketrans("ATCG", "TAGC"))[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

def random_mutation(seq, mutation_rate=0.05):
    bases = ['A','T','C','G']
    seq_list = list(seq)
    n_mut = max(1, int(len(seq)*mutation_rate))
    idxs = random.sample(range(len(seq)), n_mut)
    for i in idxs:
        orig = seq_list[i]
        choices = [b for b in bases if b != orig]
        seq_list[i] = random.choice(choices)
    return "".join(seq_list)

def kmer_shuffle(seq, k=6):
    kmers = [seq[i:i+k] for i in range(len(seq)-k+1)]
    random.shuffle(kmers)
    return " ".join(kmers)

def augment_sequence(seq):
    seq = random_mutation(seq, mutation_rate=0.05)
    return kmer_shuffle(seq)

# --- Dataset ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512, augment=False):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        if self.augment and random.random() < 0.5:
            seq = augment_sequence(seq)
        else:
            seq = kmer_tokenize(seq)
        encoding = self.tokenizer(seq, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

# --- Model ---
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_output, _ = self.attn(x, x, x)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.ff(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

class LongformerTransformerEncoder(nn.Module):
    def __init__(self, model_name="allenai/longformer-base-4096", dropout=0.3):
        super().__init__()
        self.longformer = AutoModel.from_pretrained(model_name)
        self.transformer = TransformerEncoderBlock(self.longformer.config.hidden_size, num_heads=4, dropout=dropout)
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.longformer.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.longformer(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        encoded = self.transformer(hidden_states)
        pooled = self.pooling(encoded.transpose(1, 2)).squeeze(-1)
        logits = self.classifier(self.dropout(pooled)).squeeze(-1)
        return logits

# --- Evaluation ---
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'SN': recall_score(y_true, y_pred),
        'SP': tn / (tn + fp),
        'ACC': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_prob),
        'CM': cm
    }

def plot_auc_curve(y_true, y_prob, stage, fold):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc_score(y_true, y_prob):.4f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{stage} - Fold {fold+1} ROC Curve')
    plt.legend(loc='lower right')
    plt.savefig(f'{stage}_fold{fold+1}_roc.png')
    plt.close()

def plot_loss_curve(losses, stage, fold):
    plt.figure()
    plt.plot(losses, label='Training Loss')
    plt.title(f'{stage} - Fold {fold+1} Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(f'{stage}_fold{fold+1}_loss.png')
    plt.close()

def plot_confusion_matrix(cm, stage, fold):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    plt.title(f'{stage} - Fold {fold+1} Confusion Matrix')
    plt.savefig(f'{stage}_fold{fold+1}_cm.png')
    plt.close()

# --- Data Prep ---
def prepare_stage_data(stage):
    strong = load_fasta_clean("/kaggle/input/enhancer-detection/strong_742.txt")
    weak = load_fasta_clean("/kaggle/input/enhancer-detection/weak_742.txt")
    max_len = max(len(strong), len(weak))
    strong = (strong * ((max_len // len(strong)) + 1))[:max_len]
    weak = (weak * ((max_len // len(weak)) + 1))[:max_len]
    X = strong + [reverse_complement(seq) for seq in strong] + \
        weak + [reverse_complement(seq) for seq in weak]
    y = [1]*len(strong)*2 + [0]*len(weak)*2
    return X, y

# --- Training ---
def run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔁 {stage} - Fold {fold+1}/{k}")
        train_dataset = SimpleDataset([X[i] for i in train_idx], [y[i] for i in train_idx], tokenizer, augment=True)
        val_dataset = SimpleDataset([X[i] for i in val_idx], [y[i] for i in val_idx], tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=32)

        model = LongformerTransformerEncoder()
        model = nn.DataParallel(model)
        model = model.to(device)

        optimizer = AdamW(model.parameters(), lr=5e-6)
        loss_fn = nn.BCEWithLogitsLoss()
        scaler = GradScaler()

        losses = []
        start_time = time.time()
        torch.cuda.reset_peak_memory_stats()

        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                with autocast(device_type='cuda'):
                    logits = model(input_ids, attention_mask)
                    loss = loss_fn(logits, labels)

                if torch.isnan(loss):
                    print(f"🚨 NaN loss detected at epoch {epoch+1}, fold {fold+1}. Breaking.")
                    break

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)
            losses.append(avg_loss)
            print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

        train_time = time.time() - start_time
        memory_used = torch.cuda.max_memory_allocated() / 1e6
        param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

        model.eval()
        y_true, y_pred, y_prob = [], [], []
        inference_start = time.time()
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                with autocast(device_type='cuda'):
                    logits = model(input_ids, attention_mask)
                    probs = torch.sigmoid(logits)
                y_true.extend(labels.cpu().tolist())
                y_prob.extend(probs.cpu().tolist())
                y_pred.extend((probs > 0.5).int().cpu().tolist())
        inference_time = time.time() - inference_start

        y_prob = np.array(y_prob)
        if np.isnan(y_prob).any():
            print(f"🚨 NaNs in predictions at fold {fold+1}. Skipping metric computation.")
            continue

        metrics = compute_metrics(y_true, y_pred, y_prob)
        metrics.update({
            'TrainTimeSec': train_time,
            'InferTimeSec': inference_time,
            'Params': param_count,
            'MemMB': memory_used
        })
        print(metrics)

        plot_auc_curve(y_true, y_prob, stage, fold)
        plot_loss_curve(losses, stage, fold)
        plot_confusion_matrix(metrics['CM'], stage, fold)
        results.append(metrics)

    avg = {k: np.mean([fold[k] for fold in results if k in fold]) for k in results[0]}
    print(f"\n=== Final {stage} {k}-Fold Averages ===")
    for k, v in avg.items():
        print(f"{k}: {v:.4f}")

# --- Run ---
if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained("allenai/longformer-base-4096", do_lower_case=False, trust_remote_code=True)
    X, y = prepare_stage_data("Stage2")
    run_cross_validation(X, y, "Stage2", tokenizer, epochs=5, k=10)


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


🔁 Stage2 - Fold 1/10


2025-07-13 10:00:46.089266: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752400846.269594      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752400846.320720      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/597M [00:00<?, ?B/s]

Epoch 1 Loss: 0.7013
Epoch 2 Loss: 0.6989
Epoch 3 Loss: 0.6910
Epoch 4 Loss: 0.6907
Epoch 5 Loss: 0.6877
{'SN': 0.9514824797843666, 'SP': 0.14285714285714285, 'ACC': 0.5471698113207547, 'F1': 0.6775431861804222, 'MCC': 0.16035319217147986, 'AUC': 0.5901693536082997, 'CM': array([[ 53, 318],
       [ 18, 353]]), 'TrainTimeSec': 2014.067616701126, 'InferTimeSec': 11.281350135803223, 'Params': 155748097, 'MemMB': 6678.192128}

🔁 Stage2 - Fold 2/10
Epoch 1 Loss: 0.6935
Epoch 2 Loss: 0.6867
Epoch 3 Loss: 0.6856
Epoch 4 Loss: 0.6829
Epoch 5 Loss: 0.6848
{'SN': 0.40431266846361186, 'SP': 0.7223719676549866, 'ACC': 0.5633423180592992, 'F1': 0.4807692307692307, 'MCC': 0.1336235999174286, 'AUC': 0.575577044630597, 'CM': array([[268, 103],
       [221, 150]]), 'TrainTimeSec': 2014.0118737220764, 'InferTimeSec': 11.25287938117981, 'Params': 155748097, 'MemMB': 6674.710016}

🔁 Stage2 - Fold 3/10
Epoch 1 Loss: 0.7012
Epoch 2 Loss: 0.6939
Epoch 3 Loss: 0.6915
Epoch 4 Loss: 0.6878
Epoch 5 Loss: 0.6883

LONGFORMER + CNN + BiLSTM + Attention

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef, recall_score
from collections import Counter
from tqdm import tqdm
import random, re, os, gc, time
from torch.amp import autocast, GradScaler
import seaborn as sns

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Utilities ---
def clean_sequence(seq):
    return re.sub('[^ATCG]', '', seq.upper())

def load_fasta_clean(path):
    with open(path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    return seq.translate(str.maketrans("ATCG", "TAGC"))[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

def random_mutation(seq, mutation_rate=0.05):
    bases = ['A','T','C','G']
    seq_list = list(seq)
    n_mut = max(1, int(len(seq)*mutation_rate))
    idxs = random.sample(range(len(seq)), n_mut)
    for i in idxs:
        orig = seq_list[i]
        choices = [b for b in bases if b != orig]
        seq_list[i] = random.choice(choices)
    return "".join(seq_list)

def kmer_shuffle(seq, k=6):
    kmers = [seq[i:i+k] for i in range(len(seq)-k+1)]
    random.shuffle(kmers)
    return " ".join(kmers)

def augment_sequence(seq):
    seq = random_mutation(seq, mutation_rate=0.05)
    return kmer_shuffle(seq)

# --- Dataset ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512, augment=False):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        if self.augment and random.random() < 0.5:
            seq = augment_sequence(seq)
        else:
            seq = kmer_tokenize(seq)
        encoding = self.tokenizer(seq, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

# --- Longformer + CNN + BiLSTM + Attention ---
class FeedForwardAttention(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.W = nn.Linear(input_dim, 1, bias=False)

    def forward(self, x):
        e = self.W(x).squeeze(-1)
        alpha = torch.softmax(e, dim=1).unsqueeze(-1)
        context = (alpha * x).sum(dim=1)
        return context

class LongformerCNNBiLSTM(nn.Module):
    def __init__(self, model_name="allenai/longformer-base-4096", dropout=0.3):
        super().__init__()
        self.longformer = AutoModel.from_pretrained(model_name)
        self.cnn = nn.Sequential(
            nn.Conv1d(self.longformer.config.hidden_size, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        self.lstm = nn.LSTM(self.longformer.config.hidden_size, 128, batch_first=True, bidirectional=True)
        self.attention = FeedForwardAttention(128 * 2)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(64 + 128 * 2, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.longformer(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        cnn_input = last_hidden.permute(0, 2, 1)
        cnn_features = self.cnn(cnn_input)
        cnn_features = torch.max(cnn_features, dim=2)[0]
        lstm_out, _ = self.lstm(last_hidden)
        attn_out = self.attention(lstm_out)
        combined = torch.cat([cnn_features, attn_out], dim=1)
        combined = self.dropout(combined)
        logits = self.classifier(combined).squeeze(-1)
        return logits

# --- Evaluation ---
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'SN': recall_score(y_true, y_pred),
        'SP': tn / (tn + fp),
        'ACC': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_prob),
        'CM': cm
    }

def plot_auc_curve(y_true, y_prob, stage, fold):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc_score(y_true, y_prob):.4f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{stage} - Fold {fold+1} ROC Curve')
    plt.legend(loc='lower right')
    plt.savefig(f'{stage}_fold{fold+1}_roc.png')
    plt.close()

def plot_loss_curve(losses, stage, fold):
    plt.figure()
    plt.plot(losses, label='Training Loss')
    plt.title(f'{stage} - Fold {fold+1} Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(f'{stage}_fold{fold+1}_loss.png')
    plt.close()

def plot_confusion_matrix(cm, stage, fold):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    plt.title(f'{stage} - Fold {fold+1} Confusion Matrix')
    plt.savefig(f'{stage}_fold{fold+1}_cm.png')
    plt.close()

# --- Data Prep ---
def prepare_stage_data(stage):
    strong = load_fasta_clean(f"/kaggle/input/enhancer-detection/strong_742.txt")
    weak = load_fasta_clean(f"/kaggle/input/enhancer-detection/weak_742.txt")
    max_len = max(len(strong), len(weak))
    strong = (strong * ((max_len // len(strong)) + 1))[:max_len]
    weak = (weak * ((max_len // len(weak)) + 1))[:max_len]
    X = strong + [reverse_complement(seq) for seq in strong] + weak + [reverse_complement(seq) for seq in weak]
    y = [1]*len(strong)*2 + [0]*len(weak)*2
    return X, y

# --- Training ---
def run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔁 {stage} - Fold {fold+1}/{k}")
        train_dataset = SimpleDataset([X[i] for i in train_idx], [y[i] for i in train_idx], tokenizer, augment=(stage=="Stage2"))
        val_dataset = SimpleDataset([X[i] for i in val_idx], [y[i] for i in val_idx], tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=32)

        model = LongformerCNNBiLSTM()
        model = nn.DataParallel(model)
        model = model.to(device)

        optimizer = AdamW(model.parameters(), lr=5e-6)
        loss_fn = nn.BCEWithLogitsLoss()
        scaler = GradScaler()

        losses = []
        start_time = time.time()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats(device)

        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                with autocast(device_type='cuda'):
                    logits = model(input_ids, attention_mask)
                    loss = loss_fn(logits, labels)

                if torch.isnan(loss):
                    print(f"🚨 NaN loss at epoch {epoch+1}, fold {fold+1}. Skipping epoch.")
                    break

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)
            losses.append(avg_loss)
            print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

        train_time = time.time() - start_time
        memory_used = torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else 0
        param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

        model.eval()
        y_true, y_pred, y_prob = [], [], []
        inference_start = time.time()
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                with autocast(device_type='cuda'):
                    logits = model(input_ids, attention_mask)
                    probs = torch.sigmoid(logits)
                y_true.extend(labels.cpu().tolist())
                y_prob.extend(probs.cpu().tolist())
                y_pred.extend((probs > 0.5).int().cpu().tolist())
        inference_time = time.time() - inference_start

        y_prob = np.array(y_prob)
        if np.isnan(y_prob).any():
            print(f"🚨 NaNs in predictions at fold {fold+1}. Skipping metrics.")
            continue

        metrics = compute_metrics(y_true, y_pred, y_prob)
        metrics.update({
            'TrainTimeSec': train_time,
            'InferTimeSec': inference_time,
            'Params': param_count,
            'MemMB': memory_used
        })
        print(metrics)

        plot_auc_curve(y_true, y_prob, stage, fold)
        plot_loss_curve(losses, stage, fold)
        plot_confusion_matrix(metrics['CM'], stage, fold)
        results.append(metrics)

    avg = {k: np.mean([fold[k] for fold in results if k in fold]) for k in results[0]}
    print(f"\n=== Final {stage} {k}-Fold Averages ===")
    for k, v in avg.items():
        print(f"{k}: {v:.4f}")

# --- Run ---
if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained("allenai/longformer-base-4096", do_lower_case=False, trust_remote_code=True)
    for stage in ["Stage1", "Stage2"]:
        X, y = prepare_stage_data(stage)
        run_cross_validation(X, y, stage, tokenizer, epochs=5, k=10)


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


🔁 Stage1 - Fold 1/10


2025-07-14 04:16:30.102434: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752466590.469892      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752466590.575700      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/597M [00:00<?, ?B/s]

Epoch 1 Loss: 0.6930
Epoch 2 Loss: 0.6840
Epoch 3 Loss: 0.6837
Epoch 4 Loss: 0.6812
Epoch 5 Loss: 0.6793
{'SN': 0.5067385444743935, 'SP': 0.6307277628032345, 'ACC': 0.568733153638814, 'F1': 0.5402298850574713, 'MCC': 0.13853530593531796, 'AUC': 0.5909213097841486, 'CM': array([[234, 137],
       [183, 188]]), 'TrainTimeSec': 1980.2432754039764, 'InferTimeSec': 10.9837167263031, 'Params': 149825409, 'MemMB': 6504.065536}

🔁 Stage1 - Fold 2/10
Epoch 1 Loss: 0.6911
Epoch 2 Loss: 0.6831
Epoch 3 Loss: 0.6803
Epoch 4 Loss: 0.6789
Epoch 5 Loss: 0.6780
{'SN': 0.4339622641509434, 'SP': 0.6765498652291105, 'ACC': 0.555256064690027, 'F1': 0.49386503067484655, 'MCC': 0.11391482152611854, 'AUC': 0.5711088992378724, 'CM': array([[251, 120],
       [210, 161]]), 'TrainTimeSec': 1976.0365347862244, 'InferTimeSec': 10.961992025375366, 'Params': 149825409, 'MemMB': 6511.636992}

🔁 Stage1 - Fold 3/10
Epoch 1 Loss: 0.6942
Epoch 2 Loss: 0.6862
Epoch 3 Loss: 0.6841
Epoch 4 Loss: 0.6813
Epoch 5 Loss: 0.6807
